In [ ]:
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
from sklearn.metrics import silhouette_score, adjusted_rand_score
from sklearn.preprocessing import StandardScaler

# =====================================================================
# 1. CARREGAMENTO E PRÉ-PROCESSAMENTO DO DATASET IONOSPHERE
# =====================================================================
url = "https://archive.ics.uci.edu/ml/machine-learning-databases/ionosphere/ionosphere.data"
data = pd.read_csv(url, header=None)

X_raw = data.iloc[:, :-1].values.astype(float)
y_true = data.iloc[:, -1].values           # 'g' ou 'b'
y_numeric = np.where(y_true == 'g', 1, 0)  # 'g'=1, 'b'=0

# Remover colunas constantes (variância zero)
variances = X_raw.var(axis=0)
X_raw = X_raw[:, variances > 0]

# Normalização Z-score
scaler = StandardScaler()
X = scaler.fit_transform(X_raw)

N, P = X.shape
print(f'Dataset: {N} amostras, {P} variáveis (após remoção de colunas constantes)')
print(f'Classes a priori: {np.sum(y_numeric==1)} \'good\' (g), {np.sum(y_numeric==0)} \'bad\' (b)')


In [ ]:
# =====================================================================
# 2. IMPLEMENTAÇÃO DO ALGORITMO KCM-K-GH
#
# Referência: Carvalho et al. (2018), Pattern Recognition 79:370-386
#
# Kernel Gaussiano — Eq.(9):
#   K^(s)(x_i, x_k) = exp( -1/2 * sum_j (1/s_j^2)(x_ij - x_kj)^2 )
# onde s = (s_1^2,...,s_p^2) é o vetor GLOBAL de hiperparâmetros.
#
# Função objetivo — Eq.(11):
#   J = sum_i sum_{e_k in P_i} 2*(1 - K^(s)(x_k, g_i))
#
# Algorithm 1 — etapas por iteração (ordem correta):
#   Step 1 — Eq.(14): atualiza protótipos g_i  (P e s fixos)
#   Step 2 — Eq.(16): atualiza inv_s2 = 1/s_j^2  (G e P fixos)
#   Step 3 — Eq.(18): realoca cada objeto  (G e s fixos)
#
# Inicialização (Algorithm 1, linha 5, γ=1):
#   g_i = c objetos aleatórios da base
#   1/s_j^2 = γ^(1/p) = 1 para todo j
#   Alocação inicial antes do loop
# =====================================================================

def compute_kernel(X, g, inv_s2):
    """Matriz de kernel N x c.  inv_s2[j] = 1/s_j^2."""
    N, c = X.shape[0], len(g)
    K = np.zeros((N, c))
    for k in range(c):
        diff2 = (X - g[k]) ** 2        # (N, P)
        K[:, k] = np.exp(-0.5 * (diff2 @ inv_s2))  # (N,)
    return K


def kcm_k_gh(X, c, max_iter=300, eps=1e-10, seed=None):
    """
    Executa KCM-K-GH conforme Algorithm 1 do artigo.

    Retorna:
        g      : protótipos (c, P)
        s2     : vetor s_j^2  (P,)  — notação do artigo
        labels : partição crisp (N,)
        J_hist : lista de J por iteração
    """
    rng = np.random.default_rng(seed)
    N, P = X.shape

    # ---- INICIALIZAÇÃO ----
    idx = rng.choice(N, c, replace=False)
    g = X[idx].copy()
    inv_s2 = np.ones(P)      # 1/s_j^2 = 1 (γ=1)

    # Alocação inicial (linha 7 do Algorithm 1)
    K = compute_kernel(X, g, inv_s2)
    labels = np.argmax(K, axis=1)
    for k in range(c):
        if np.sum(labels == k) == 0:
            labels[rng.integers(N)] = k

    J_hist = []

    for _ in range(max_iter):
        old_labels = labels.copy()

        # --- Step 1: Representação — Eq.(14) ---
        # K com g e inv_s2 atuais (antes de atualizar g)
        K = compute_kernel(X, g, inv_s2)
        for k in range(c):
            mask = labels == k
            if not mask.any():
                g[k] = X[rng.integers(N)]
                continue
            w = K[mask, k]
            g[k] = (X[mask].T @ w) / (w.sum() + eps)  # Eq.(14)

        # --- Step 2: Larguras — Eq.(16) com γ=1 ---
        # Recalcula K com os NOVOS protótipos g
        K = compute_kernel(X, g, inv_s2)
        D = np.zeros(P)
        for k in range(c):
            mask = labels == k
            if not mask.any():
                continue
            w = K[mask, k]               # (n_k,)
            diff2 = (X[mask] - g[k]) ** 2  # (n_k, P)
            D += diff2.T @ w             # Numerador de D_j

        # 1/s_j^2 = geomean(D) / D_j  (Eq.16, γ=1)
        D_safe = np.maximum(D, eps)
        log_geomean = np.mean(np.log(D_safe))
        inv_s2 = np.exp(log_geomean - np.log(D_safe))  # (P,)

        # --- Step 3: Alocação — Eq.(18) ---
        # Recalcula K com novos g E novo inv_s2
        K = compute_kernel(X, g, inv_s2)
        labels = np.argmax(K, axis=1)
        for k in range(c):
            if np.sum(labels == k) == 0:
                g[k] = X[rng.integers(N)]
                labels[rng.integers(N)] = k

        # Função objetivo J — Eq.(11)
        J = 2.0 * np.sum(1.0 - K[np.arange(N), labels])
        J_hist.append(J)

        if np.array_equal(labels, old_labels):
            break

    return g, 1.0 / inv_s2, labels, J_hist  # s_j^2 = 1/inv_s2


In [ ]:
# =====================================================================
# 3. LOOP EXPERIMENTAL — 100 execuções por c ∈ {2,3,4,5,6}
# =====================================================================
c_values = [2, 3, 4, 5, 6]
best_results = {}    # c -> (g, s2, labels, J_hist)
silhouette_scores = []

print('Iniciando as replicações (100 rodadas por c)...')
print('-' * 55)

for c in c_values:
    best_J = np.inf
    best_run = None

    for run in range(100):
        g, s2, labels, J_hist = kcm_k_gh(X, c)
        J_final = J_hist[-1]
        if J_final < best_J:
            best_J = J_final
            best_run = (g.copy(), s2.copy(), labels.copy(), list(J_hist))

    best_results[c] = best_run
    sil = silhouette_score(X, best_run[2])
    silhouette_scores.append(sil)
    print(f'  c={c}  |  Melhor J = {best_J:.4f}  |  Silhueta = {sil:.4f}')

print('-' * 55)
print('Concluído.')


In [ ]:
# =====================================================================
# 4. PLOT SILHUETA × c  e  escolha de c*
# =====================================================================
plt.figure(figsize=(7, 4))
plt.plot(c_values, silhouette_scores, marker='o', color='steelblue', linewidth=2)
plt.title('Coeficiente de Silhueta vs Número de Clusters (KCM-K-GH)')
plt.xlabel('Número de clusters (c)')
plt.ylabel('Silhueta média')
plt.xticks(c_values)
plt.grid(True, linestyle='--', alpha=0.6)
plt.tight_layout()
plt.show()

c_opt = c_values[int(np.argmax(silhouette_scores))]
print(f'c* = argmax_c Sil(c) = {c_opt}')
print(f'Silhueta para c*={c_opt}: {max(silhouette_scores):.4f}')


In [ ]:
# =====================================================================
# 5. ÍNDICE DE RAND CORRIGIDO (ARI)  +  COMENTÁRIO
# =====================================================================
g_opt, s2_opt, labels_opt, J_hist_opt = best_results[c_opt]

ari = adjusted_rand_score(y_numeric, labels_opt)
print('=' * 60)
print(f' Resultados para c* = {c_opt}')
print('=' * 60)
print(f'\nÍndice de Rand Corrigido (ARI): {ari:.4f}')

print()
print('Comentário:')
if ari > 0.3:
    print(f'  ARI = {ari:.4f} indica concordância moderada a boa entre a partição')
    print('  obtida pelo KCM-K-GH e os rótulos originais do Ionosphere.')
    print('  O algoritmo, mesmo sendo não-supervisionado, conseguiu capturar')
    print('  parte da estrutura discriminativa dos dados.')
elif ari > 0.05:
    print(f'  ARI = {ari:.4f} indica baixa concordância entre a partição gerada')
    print('  pelo KCM-K-GH e os rótulos originais. Os clusters encontrados')
    print('  refletem a estrutura geométrica no espaço kernelizado, que não')
    print('  coincide plenamente com a partição a priori.')
else:
    print(f'  ARI ≈ {ari:.4f}. A partição gerada não tem correspondência')
    print('  significativa com os rótulos a priori. O algoritmo de clusterização')
    print('  organiza os dados segundo a estrutura geométrica no espaço')
    print('  kernelizado, o que não reflete necessariamente as classes originais.')


In [ ]:
# =====================================================================
# 6.i) PROTÓTIPOS DE CADA GRUPO  g_1, ..., g_{c*}
# =====================================================================
print(f'i) Protótipos para c* = {c_opt} (espaço normalizado):')
print('-' * 60)
for idx, gi in enumerate(g_opt):
    print(f'  g_{idx+1} = {np.round(gi, 4)}')


In [ ]:
# =====================================================================
# 6.ii) VETOR DE PARÂMETROS DE LARGURA  s = (s_1^2, ..., s_p^2)
#
# Notação do artigo (Eq.9): o kernel usa 1/s_j^2.
# s_j^2 pequeno → kernel estreito → variável mais relevante.
# s_j^2 grande  → kernel largo   → variável menos relevante.
# =====================================================================
print(f'ii) Vetor de parâmetros de largura s_j^2  (j=1,...,{P}):')
print('-' * 60)
print(np.round(s2_opt, 6))

top5_rel = np.argsort(s2_opt)[:5] + 1
top5_irr = np.argsort(s2_opt)[-5:] + 1
print(f'\n  5 variáveis mais relevantes  (menor s_j^2): {top5_rel}')
print(f'  5 variáveis menos relevantes (maior s_j^2): {top5_irr}')


In [ ]:
# =====================================================================
# 6.iii) MATRIZ DE CONFUSÃO — Clusters do algoritmo vs classes a priori
# Linhas = classes a priori (0='bad', 1='good')
# Colunas = clusters do algoritmo (0, ..., c*-1)
# =====================================================================
print(f'iii) Matriz de confusão: classes a priori (linhas) vs clusters (colunas)')
print('-' * 60)

cm = np.zeros((2, c_opt), dtype=int)
for cls in range(2):
    for clu in range(c_opt):
        cm[cls, clu] = int(np.sum((y_numeric == cls) & (labels_opt == clu)))

cm_df = pd.DataFrame(
    cm,
    index=["Classe 0 (bad)", "Classe 1 (good)"],
    columns=[f"Cluster {j}" for j in range(c_opt)]
)
print(cm_df.to_string())
print()

print('Distribuição por cluster:')
for j in range(c_opt):
    total = int(np.sum(labels_opt == j))
    print(f'  Cluster {j}: {total:3d} objetos  '
          f'({cm[1,j]} good = {100*cm[1,j]/max(total,1):.1f}%,  '
          f'{cm[0,j]} bad = {100*cm[0,j]/max(total,1):.1f}%)')


In [ ]:
# =====================================================================
# 6.iv) PLOT DA FUNÇÃO OBJETIVO vs ITERAÇÕES
# =====================================================================
plt.figure(figsize=(7, 4))
iters = list(range(1, len(J_hist_opt) + 1))
plt.plot(iters, J_hist_opt, marker='o', markersize=4, color='crimson', linewidth=2)
plt.title(f'Convergência da Função Objetivo J  (c* = {c_opt},  melhor de 100 runs)')
plt.xlabel('Iteração')
plt.ylabel('J  (Eq. 11)')
plt.grid(True, linestyle='--', alpha=0.6)
plt.tight_layout()
plt.show()

print(f'Iterações até convergência : {len(J_hist_opt)}')
print(f'J inicial  : {J_hist_opt[0]:.4f}')
print(f'J final    : {J_hist_opt[-1]:.4f}')
print(f'Redução    : {J_hist_opt[0]-J_hist_opt[-1]:.4f}  ({100*(J_hist_opt[0]-J_hist_opt[-1])/J_hist_opt[0]:.1f}%)')
